# Exploratory data analysis of the Abalone dataset

This notebook explores the structure, quality, distributions, and relationships in the Abalone dataset before modelling the number of shell rings. The analysis keeps the original observations unchanged and creates `Age` only as a derived interpretation of the target.

## Imports and display settings

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")
sns.set_theme(style="whitegrid", context="notebook")

## Load and validate the data

The path works when the notebook is launched either from the repository root or from the `notebooks` directory. The schema check fails early if a different CSV is used by mistake.

In [ ]:
data_path = Path("data/abalone.csv")
if not data_path.exists():
    data_path = Path("../data/abalone.csv")

expected_columns = [
    "Sex",
    "Length",
    "Diameter",
    "Height",
    "Whole weight",
    "Shucked weight",
    "Viscera weight",
    "Shell weight",
    "Rings",
]

df = pd.read_csv(data_path)
assert df.columns.tolist() == expected_columns, "Unexpected dataset schema"
print(f"Dataset: {data_path.resolve()}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## Structure and data quality

In [ ]:
quality_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(),
    }
)
quality_summary

In [ ]:
duplicate_count = int(df.duplicated().sum())
invalid_sex_count = int((~df["Sex"].isin(["F", "I", "M"])).sum())
non_positive_measurements = (df.select_dtypes(include=np.number) <= 0).sum()

print(f"Duplicate rows: {duplicate_count}")
print(f"Unexpected Sex values: {invalid_sex_count}")
print("Non-positive numeric values:")
display(non_positive_measurements[non_positive_measurements > 0].to_frame("count"))

The dataset has no missing or duplicate rows. Two observations have a height of zero, which is physically implausible. Very large height values also deserve review before modelling. They are flagged below rather than removed, so the raw data remains intact.

In [ ]:
height_flags = df.loc[(df["Height"] <= 0) | (df["Height"] > 0.30)].copy()
height_flags

## Descriptive statistics

Abalone age is commonly approximated as the number of rings plus 1.5 years. `Age` is therefore derived from `Rings` and must never be used as an input feature when predicting `Rings`, because that would create target leakage.

In [ ]:
eda_df = df.assign(Age=df["Rings"] + 1.5)
eda_df.describe().T

## Target distribution

In [ ]:
rings_summary = eda_df["Rings"].describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
display(rings_summary.to_frame("Rings"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=eda_df, x="Rings", bins=29, discrete=True, ax=axes[0], color="#2563EB")
axes[0].axvline(eda_df["Rings"].median(), color="#DC2626", linestyle="--", label="Median")
axes[0].set_title("Distribution of the target")
axes[0].legend()
sns.boxplot(data=eda_df, x="Rings", ax=axes[1], color="#93C5FD")
axes[1].set_title("Target spread and high-end outliers")
plt.tight_layout()
plt.show()

`Rings` is concentrated around 8 to 11 and has a longer right tail. A model evaluated only with RMSE may be strongly affected by the relatively rare older abalones, so MAE should also be reported.

## Numeric feature distributions

In [ ]:
numeric_features = [column for column in expected_columns if column not in {"Sex", "Rings"}]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, column in zip(axes.flat, numeric_features):
    sns.histplot(data=eda_df, x=column, bins=30, kde=True, ax=ax, color="#0F766E")
    ax.set_title(column)
for ax in axes.flat[len(numeric_features) :]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

The weight variables are right-skewed, while length and diameter are left-skewed. Height is strongly affected by its extreme values. Standardization aligns feature scales, which matters for regularized linear and distance-based models; it does not remove skewness. Tree-based models are less sensitive to feature scale.

## Sex category and target

In [ ]:
sex_summary = (
    eda_df.groupby("Sex", observed=True)
    .agg(
        observations=("Rings", "size"),
        mean_rings=("Rings", "mean"),
        median_rings=("Rings", "median"),
        mean_whole_weight=("Whole weight", "mean"),
    )
    .round(3)
)
display(sex_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=eda_df, x="Sex", order=["F", "I", "M"], ax=axes[0], color="#2563EB")
axes[0].set_title("Observations by sex category")
sns.boxplot(data=eda_df, x="Sex", y="Rings", order=["F", "I", "M"], ax=axes[1], color="#93C5FD")
axes[1].set_title("Rings by sex category")
plt.tight_layout()
plt.show()

The categories are reasonably balanced. Infant (`I`) abalones have fewer rings on average and are also lighter, so the observed group difference should not be interpreted as an isolated causal effect of sex. `Sex` will require categorical encoding for modelling.

## Relationships with the target

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, column in zip(axes.flat, numeric_features):
    sns.scatterplot(
        data=eda_df,
        x=column,
        y="Rings",
        hue="Sex",
        hue_order=["F", "I", "M"],
        alpha=0.25,
        s=15,
        linewidth=0,
        legend=True,
        ax=ax,
    )
    ax.set_title(column)
for ax in axes.flat[len(numeric_features) :]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

All physical measurements have a positive relationship with `Rings`, but the spread grows for larger abalones and the relationship is not strictly linear. This supports comparing a linear baseline with a nonlinear tree-based model.

## Correlations and multicollinearity

In [ ]:
correlation_matrix = eda_df.drop(columns="Age").corr(numeric_only=True)
target_correlations = correlation_matrix["Rings"].drop("Rings").sort_values(ascending=False)
display(target_correlations.to_frame("correlation_with_rings"))

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

`Shell weight` has the strongest individual linear correlation with `Rings`. Most size and weight features are also highly correlated with each other, especially length with diameter and whole weight with its component weights. Coefficients from an unregularized linear model may therefore be unstable even when predictive performance is acceptable.

## IQR-based outlier review

The IQR rule is used only to flag observations for review. Extreme values may represent valid old or large abalones, so they should not be removed without domain justification.

**What is the IQR?** The interquartile range is the spread of the middle 50% of the data: `IQR = Q3 - Q1`, where Q1 is the 25th percentile and Q3 the 75th percentile.

**The rule.** A value is flagged as an outlier if it is outside these bounds:

- lower bound = `Q1 - 1.5 * IQR`
- upper bound = `Q3 + 1.5 * IQR`

This is the same rule used to draw the whiskers of a boxplot.

**How to read the table below.** For each feature, `lower_bound` and `upper_bound` are the limits, `outlier_count` is the number of rows outside them and `outlier_pct` is that count as a percentage of all rows. A high percentage is a reason to look at the data, not a reason to delete it.

In [ ]:
outlier_rows = []
for column in numeric_features + ["Rings"]:
    q1, q3 = eda_df[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = int(((eda_df[column] < lower_bound) | (eda_df[column] > upper_bound)).sum())
    outlier_rows.append(
        {
            "feature": column,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "outlier_count": outlier_count,
            "outlier_pct": 100 * outlier_count / len(eda_df),
        }
    )

outlier_summary = pd.DataFrame(outlier_rows).set_index("feature").round(3)
outlier_summary

## Conclusions and modelling implications

- The dataset contains 4,177 observations, eight predictors, and the `Rings` target. No missing or duplicate rows were found.
- Two zero heights and two unusually large heights require review. They remain in the raw data at this stage.

<br>

- `Rings` is right-skewed, with most observations between 8 and 11. MAE and RMSE should both be reported because RMSE penalizes large prediction errors more strongly, which may be relevant for rare high-ring observations.

<br>

- Infant abalones have fewer rings and lower weight on average. `Sex` should be one-hot encoded, while group differences should not be interpreted causally.
- `Shell weight` is the strongest single linear correlate of `Rings`. Physical measurements are strongly correlated with each other, which creates multicollinearity.
- The feature-target relationships show nonlinearity and changing variance. A regularized linear model is a useful comparison to ordinary linear regression. A nonlinear tree-based regressor is an optional future comparison; the assignment permits a simple linear regression for this first version.

<br>

- `Age` is derived directly from `Rings` and must not be included among the model inputs.